In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Техніки пом'якшення та придушення помилок

> **Note:** Бета-версія нової моделі виконання вже доступна. Спрямована модель виконання забезпечує більшу гнучкість при налаштуванні робочого процесу пом'якшення помилок. Дивись посібник [Спрямована модель виконання](/guides/directed-execution-model) для отримання додаткової інформації.


<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці було розроблено з використанням наступних вимог.
Ми рекомендуємо використовувати ці або новіші версії.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Техніки пом'якшення та придушення помилок використовуються для покращення якості результатів при масштабуванні до більших робочих навантажень. На цій сторінці наведено пояснення технік придушення та пом'якшення помилок, доступних через Qiskit Runtime.

Наступна комірка імпортує примітив Estimator і створює backend, який використовуватиметься для ініціалізації Estimator у подальших комірках коду.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Динамічне роз'єднання
Квантові схеми виконуються на апаратному забезпеченні IBM&reg; у вигляді послідовностей мікрохвильових імпульсів, які потрібно планувати та запускати в точні часові інтервали.
На жаль, небажані взаємодії між кубітами можуть призводити до когерентних помилок на простоюючих кубітах. Динамічне роз'єднання працює шляхом вставки послідовностей імпульсів на простоюючих кубітах, щоб приблизно скасувати ефект цих помилок. Кожна вставлена послідовність імпульсів є операцією тотожності, але фізична наявність імпульсів має ефект придушення помилок.
Існує багато можливих варіантів послідовностей імпульсів, і те, яка послідовність краща для кожного конкретного випадку, залишається [активною областю досліджень](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Зверни увагу, що динамічне роз'єднання в основному корисне для схем, що містять проміжки, у яких деякі кубіти простоюють без будь-яких операцій над ними. Якщо операції в схемі розміщені дуже щільно, так що всі кубіти більшість часу зайняті, то додавання імпульсів динамічного роз'єднання може не покращити продуктивність. Насправді це може навіть погіршити продуктивність через недосконалості самих імпульсів.

Діаграма нижче зображує динамічне роз'єднання з послідовністю імпульсів XX. Абстрактна схема ліворуч відображається на розклад мікрохвильових імпульсів у верхньому правому куті. У нижньому правому куті зображено той самий розклад, але з послідовністю двох X-імпульсів, вставленою під час простою першого кубіта.

![Зображення динамічного роз'єднання](../docs/images/guides/error-mitigation-explanation/dd.avif)

Динамічне роз'єднання можна ввімкнути, встановивши `enable` на `True` у [параметрах динамічного роз'єднання](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). Параметр `sequence_type` можна використовувати для вибору з декількох різних послідовностей імпульсів. Тип послідовності за замовчуванням — `"XX"`.

Наступна комірка коду показує, як увімкнути динамічне роз'єднання для Estimator і вибрати послідовність динамічного роз'єднання.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Скручування Паулі
Скручування, також відоме як [рандомізована компіляція](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), є широко вживаною технікою для перетворення довільних каналів шуму на канали шуму з більш конкретною структурою.

Скручування Паулі — це особливий вид скручування, що використовує операції Паулі. Воно має ефект перетворення будь-якого квантового каналу на канал Паулі. Виконуване саме по собі, воно може пом'якшити когерентний шум, оскільки когерентний шум, як правило, накопичується квадратично з кількістю операцій, тоді як шум Паулі накопичується лінійно. Скручування Паулі часто поєднується з іншими техніками пом'якшення помилок, які краще працюють із шумом Паулі, ніж із довільним шумом.

Скручування Паулі реалізується шляхом «обгортання» вибраного набору гейтів випадково обраними однокубітними гейтами Паулі таким чином, що ідеальний ефект гейта залишається незмінним. У результаті одна схема замінюється випадковим ансамблем схем, усі з однаковим ідеальним ефектом. При вибірці схеми зразки беруться з кількох випадкових екземплярів, а не лише з одного.

![Зображення скручування Паулі](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Оскільки більшість помилок у сучасному квантовому апаратному забезпеченні походить від двокубітних гейтів, ця техніка часто застосовується виключно до (нативних) двокубітних гейтів. Наступна діаграма зображує деякі скручування Паулі для гейтів CNOT і ECR. Кожна схема в рядку має однаковий ідеальний ефект.

![Зображення скручувань гейтів](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

Скручування Паулі можна ввімкнути, встановивши `enable_gates` на `True` у [параметрах скручування](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Інші важливі параметри:

- `num_randomizations`: Кількість екземплярів схем, які беруться з ансамблю скручених схем.
- `shots_per_randomization`: Кількість вимірювань для кожного екземпляра схеми.

Наступна комірка коду показує, як увімкнути скручування Паулі та встановити ці параметри для estimator. Жоден із цих параметрів не потрібно встановлювати явно.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Скручуване усунення помилок зчитування (TREX)
[Скручуване усунення помилок зчитування (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) пом'якшує вплив помилок вимірювання при оцінці очікуваних значень спостережуваних операторів Паулі.
Воно базується на понятті скручуваних вимірювань, що досягаються шляхом випадкової заміни гейтів вимірювання послідовністю з (1) гейта Паулі X, (2) вимірювання та (3) класичного перемикання біта. Як і при стандартному скручуванні гейтів, ця послідовність еквівалентна простому вимірюванню за відсутності шуму, як зображено на наступній діаграмі:

![Зображення скручування вимірювань](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

За наявності помилки зчитування скручування вимірювань має ефект діагоналізації матриці переносу помилок зчитування, що робить її легкою для інвертування. Оцінка матриці переносу помилок зчитування потребує виконання додаткових калібрувальних схем, що вносить невеликі накладні витрати.

TREX можна ввімкнути, встановивши `measure_mitigation` на `True` у [параметрах відмовостійкості Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) для Estimator. Параметри навчання шуму вимірювань описані [тут](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Як і при скручуванні гейтів, можна встановити кількість рандомізацій схем та кількість вимірювань на рандомізацію.

Наступна комірка коду показує, як увімкнути TREX та встановити ці параметри для estimator. Жоден із цих параметрів не потрібно встановлювати явно.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Екстраполяція до нульового шуму (ZNE)
Екстраполяція до нульового шуму (ZNE) — це техніка пом'якшення помилок при оцінці очікуваних значень спостережуваних операторів. Хоча вона часто покращує результати, вона не гарантує отримання незміщеного результату.

ZNE складається з двох етапів:

1. _Підсилення шуму_: Оригінальна квантова схема виконується кілька разів при різних рівнях шуму.
2. _Екстраполяція_: Ідеальний результат оцінюється шляхом екстраполяції шумних очікуваних значень до межі нульового шуму.

Обидва етапи — підсилення шуму та екстраполяція — можуть бути реалізовані різними способами. Qiskit Runtime реалізує підсилення шуму за допомогою «цифрового складання гейтів», що означає заміну двокубітних гейтів еквівалентними послідовностями гейта та його оберненого. Наприклад, заміна унітарного $U$ на $U U^\dagger U$ дасть коефіцієнт підсилення шуму 3. Для екстраполяції можна вибрати одну з кількох функціональних форм, включаючи лінійну або експоненційну апроксимацію.
На зображенні нижче ліворуч зображено цифрове складання гейтів, а праворуч — процедуру екстраполяції.

![Зображення ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

ZNE можна ввімкнути, встановивши `zne_mitigation` на `True` у [параметрах відмовостійкості Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) для Estimator.
Параметри Qiskit Runtime для ZNE описані [тут](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Наступні параметри є важливими:

- `noise_factors`: Коефіцієнти шуму для підсилення шуму.
- `extrapolator`: Функціональна форма для екстраполяції.

Наступна комірка коду показує, як увімкнути ZNE та встановити ці параметри для estimator. Жоден із цих параметрів не потрібно встановлювати явно.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Імовірнісне підсилення помилок (PEA)
Одним із головних викликів у ZNE є точне підсилення шуму, що впливає на цільову схему. Складання гейтів забезпечує простий спосіб виконання цього підсилення, але потенційно є неточним і може призводити до неправильних результатів. Дивись статтю ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), зокрема стор. 4 додаткової інформації для деталей. Імовірнісне підсилення помилок забезпечує більш точний підхід до підсилення помилок через навчання шуму.

PEA — це більш складна техніка, яка проводить попередні експерименти для реконструкції шуму, а потім використовує цю інформацію для виконання точного підсилення. Вона починає з навчання скрученої моделі шуму кожного шару заплутувальних гейтів у схемі перед їх виконанням (дивись [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) для відповідних параметрів навчання). Після фази навчання схеми виконуються при кожному коефіцієнті шуму, де кожен заплутувальний шар схем підсилюється шляхом імовірнісного введення однокубітного шуму, пропорційного відповідній вивченій моделі шуму. Дивись статтю ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) для отримання додаткової інформації.

PEA складається з трьох етапів:
1. _Навчання_: Вивчається скручена модель шуму кожного шару заплутувальних гейтів у схемі.
1. _Підсилення шуму_: Оригінальна квантова схема виконується кілька разів при різних коефіцієнтах шуму.
2. _Екстраполяція_: Ідеальний результат оцінюється шляхом екстраполяції шумних очікуваних значень до межі нульового шуму.

Для експериментів у масштабі реальних задач PEA часто є найкращим вибором.

Оскільки PEA є технікою підсилення шуму ZNE, також потрібно ввімкнути ZNE, встановивши `resilience.zne_mitigation = True`. Додаткові параметри [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) можна використовувати для встановлення екстраполяторів, рівнів підсилення тощо. PEA потребує моделі шуму, яка автоматично генерується при використанні примітивів.

Наступний фрагмент надає приклад, де PEA використовується для пом'якшення результату завдання Estimator:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Імовірнісне скасування помилок (PEC)
Імовірнісне скасування помилок (PEC) — це техніка пом'якшення помилок при оцінці очікуваних значень спостережуваних операторів. На відміну від ZNE, вона повертає незміщену оцінку очікуваного значення. Однак вона, як правило, несе більші накладні витрати.

У PEC ефект ідеальної цільової схеми виражається як лінійна комбінація шумних схем, які насправді можна реалізувати на практиці:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

Вихідні дані ідеальної схеми можна потім відтворити, виконуючи різні шумні екземпляри схем, взяті з випадкового ансамблю, визначеного лінійною комбінацією. Якщо коефіцієнти $\eta_i$ утворюють розподіл імовірностей, їх можна використовувати безпосередньо як імовірності ансамблю. На практиці деякі коефіцієнти є від'ємними, тому вони утворюють квазіімовірнісний розподіл. Їх все одно можна використовувати для визначення випадкового ансамблю, але існують накладні витрати на вибірку, пов'язані з від'ємністю квазіімовірнісного розподілу, що характеризується величиною

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

Накладні витрати на вибірку є мультиплікативним коефіцієнтом для кількості вимірювань, необхідних для оцінки очікуваного значення з заданою точністю, порівняно з кількістю вимірювань, яка знадобилася б для ідеальної схеми. Вони масштабуються квадратично з $\gamma$, що, у свою чергу, масштабується експоненційно з глибиною схеми.

PEC можна ввімкнути, встановивши `pec_mitigation` на `True` у [параметрах відмовостійкості Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) для Estimator.
Параметри Qiskit Runtime для PEC описані [тут](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). Обмеження накладних витрат на вибірку можна встановити за допомогою параметра `max_overhead`. Зверни увагу, що обмеження накладних витрат на вибірку може призвести до того, що точність результату перевищуватиме запитувану точність. Значення `max_overhead` за замовчуванням — 100.

Наступна комірка коду показує, як увімкнути PEC та встановити параметр `max_overhead` для estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Наступні кроки
- Ознайомся з [навчальним посібником](/tutorials/combine-error-mitigation-techniques) про поєднання параметрів пом'якшення помилок із примітивом Estimator.
- [Налаштуй пом'якшення помилок.](configure-error-mitigation)
- [Налаштуй придушення помилок.](configure-error-suppression)
- Досліди інші [параметри](runtime-options-overview) для примітивів Qiskit Runtime.
- Вибери, в якому [режимі виконання](execution-modes) запустити своє завдання.